In [1]:
!pip install safetensors transformers rich tyro
!pip uninstall protobuf
!pip install protobuf==3.20.3



[notice] A new release of pip is available: 23.3.1 -> 25.0.1
[notice] To update, run: python -m pip install --upgrade pip
Found existing installation: protobuf 3.20.3
Uninstalling protobuf-3.20.3:
  Would remove:
    /usr/local/lib/python3.10/dist-packages/google/protobuf/*
    /usr/local/lib/python3.10/dist-packages/protobuf-3.20.3-py3.10-nspkg.pth
    /usr/local/lib/python3.10/dist-packages/protobuf-3.20.3.dist-info/*
Proceed (Y/n)? ^C
ERROR: Operation cancelled by user

[notice] A new release of pip is available: 23.3.1 -> 25.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
# Code to install Unsloth, Triton, Torch etc
!pip install --no-deps bitsandbytes accelerate xformers==0.0.29 peft trl triton
!pip install --no-deps cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
!pip install --no-deps unsloth


[notice] A new release of pip is available: 23.3.1 -> 25.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 23.3.1 -> 25.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 23.3.1 -> 25.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 23.3.1 -> 25.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [3]:
# Helpful functions used through the entire notebook
import torch
import torch.nn as nn
from transformers import set_seed
import time
import inspect
import os
major_version, minor_version = torch.cuda.get_device_capability()
HAS_BFLOAT16 = (major_version >= 8)
from inspect import currentframe as _C, getframeinfo
_F = lambda c: getframeinfo(c).lineno # Gets line number
WARN = lambda x: print(f"\033[31m{x}\033[0m") # Red colored warnings

# https://stackoverflow.com/questions/18425225/getting-the-name-of-a-variable-as-a-string
def NAME(var):
    callers_local_vars = inspect.currentframe().f_back.f_locals.items()
    names = [var_name for var_name, var_val in callers_local_vars if var_val is var]
    return names[0] if len(names) != 0 else ""

def assert_same(x, y, line, dtype):
    assert(x.dtype == dtype)
    try: torch.testing.assert_close(x, y, check_stride = True)
    except Exception as error:
        raise RuntimeError(
            f"Failed allclose at line [{line}]: {NAME(x)}, {NAME(y)}\n{str(error)}"
        )

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [7]:
import torch
from torch.nn import CrossEntropyLoss
import torch.nn as nn
import torch.nn.functional as F

# ----------------------------------------------------------------------
# MemoryEfficientLinear:
# A custom autograd Function that avoids storing large intermediate logits.
class MemoryEfficientLinear(torch.autograd.Function):

    @staticmethod
    def forward(ctx, X, weight, bias, labels, chunk_size=8192):
        # Save necessary tensors for backward
        ctx.save_for_backward(X, weight, bias, labels)
        ctx.chunk_size = chunk_size
        
        # Cast to float32 for numerical stability if needed
        compute_dtype = torch.float32 if X.dtype == torch.float16 else X.dtype
        X = X.to(compute_dtype)
        weight = weight.to(compute_dtype)
        if bias is not None:
            bias = bias.to(compute_dtype)

        # Flatten input and labels
        X_flat = X.view(-1, X.size(-1)) if X.dim() > 2 else X
        labels_flat = labels.view(-1) if labels.dim() > 1 else labels
        N, D = X_flat.size()
        V = weight.size(0)

        # First pass: compute global max for numerical stability
        global_max = torch.zeros(N, device=X.device, dtype=compute_dtype)
        for start in range(0, V, chunk_size):
            end = min(start + chunk_size, V)
            logits = F.linear(X_flat, weight[start:end], bias[start:end] if bias is not None else None)
            global_max = torch.maximum(global_max, logits.max(dim=1).values)

        # Second pass: compute logsumexp and target logits
        logsumexp = torch.zeros(N, device=X.device, dtype=compute_dtype)
        target_logits = torch.zeros(N, device=X.device, dtype=compute_dtype)
        
        for start in range(0, V, chunk_size):
            end = min(start + chunk_size, V)
            logits = F.linear(X_flat, weight[start:end], bias[start:end] if bias is not None else None)
            
            # Compute logsumexp components
            shifted_logits = logits - global_max.unsqueeze(1)
            logsumexp += torch.exp(shifted_logits).sum(dim=1)
            
            # Gather target logits
            mask = (labels_flat >= start) & (labels_flat < end)
            if mask.any():
                target_logits[mask] = logits[mask, labels_flat[mask] - start]

        # Final loss computation
        logsumexp = global_max + torch.log(logsumexp)
        losses = logsumexp - target_logits
        loss = losses.mean()
        
        return loss.to(X.dtype)

    @staticmethod
    def backward(ctx, grad_output):
        X, weight, bias, labels = ctx.saved_tensors
        chunk_size = ctx.chunk_size
        compute_dtype = X.dtype
        
        X_flat = X.view(-1, X.size(-1)) if X.dim() > 2 else X
        labels_flat = labels.view(-1)
        N, D = X_flat.size()
        V = weight.size(0)

        with torch.enable_grad():
            # First pass: recompute global max
            global_max = torch.zeros(N, device=X.device, dtype=compute_dtype)
            for start in range(0, V, chunk_size):
                end = min(start + chunk_size, V)
                logits = F.linear(X_flat, weight[start:end], bias[start:end] if bias is not None else None)
                global_max = torch.maximum(global_max, logits.max(dim=1).values)

            # Gradient computation
            grad_X = torch.zeros_like(X_flat)
            grad_weight = torch.zeros_like(weight)
            grad_bias = torch.zeros_like(bias) if bias is not None else None
            
            for start in range(0, V, chunk_size):
                end = min(start + chunk_size, V)
                weight_chunk = weight[start:end].detach().requires_grad_(True)
                bias_chunk = bias[start:end].detach().requires_grad_(True) if bias is not None else None
                
                with torch.enable_grad():
                    logits = F.linear(X_flat, weight_chunk, bias_chunk)
                    mask = (labels_flat >= start) & (labels_flat < end)
                    
                    # Compute target logits
                    target_logits = torch.zeros(N, device=X.device, dtype=compute_dtype)
                    if mask.any():
                        target_logits[mask] = logits[mask, labels_flat[mask] - start]
                    
                    # Compute logsumexp for this chunk
                    shifted_logits = logits - global_max.unsqueeze(1)
                    logsumexp_chunk = torch.log(torch.exp(shifted_logits).sum(dim=1))
                    
                    # Compute chunk loss
                    loss_chunk = (global_max + logsumexp_chunk - target_logits).mean()

                # Dynamic gradient inputs based on requires_grad
                inputs = [weight_chunk]
                if bias_chunk is not None:
                    inputs.append(bias_chunk)
                if X_flat.requires_grad:
                    inputs.append(X_flat)
                
                grads = torch.autograd.grad(
                    loss_chunk,
                    inputs,
                    allow_unused=True,
                    retain_graph=True
                )
                
                # Unpack gradients
                grad_weight_chunk = grads[0]
                grad_bias_chunk = grads[1] if bias_chunk is not None else None
                grad_X_chunk = grads[-1] if X_flat.requires_grad else None
                
                # Accumulate gradients
                grad_weight[start:end] += grad_weight_chunk
                if grad_bias is not None and grad_bias_chunk is not None:
                    grad_bias[start:end] += grad_bias_chunk
                if grad_X_chunk is not None:
                    grad_X += grad_X_chunk

        # Reshape and scale gradients
        if X.dim() > 2:
            grad_X = grad_X.view(*X.shape)
            
        return (
            grad_X * grad_output if X.requires_grad else None,
            grad_weight * grad_output,
            grad_bias * grad_output if grad_bias is not None else None,
            None,
            None
        )

# ----------------------------------------------------------------------
# A default transformation function that materializes full logits.
# (For smaller inputs only.)
def transformation_function(X, linear, labels):
    # Ensure proper dimension handling
    original_shape = X.shape
    X_flat = X.view(-1, X.size(-1))
    labels_flat = labels.view(-1)
    
    # Calculate loss with flattened inputs
    return MemoryEfficientLinear.apply(
        X_flat, 
        linear.weight, 
        linear.bias if linear.bias is not None else None, 
        labels_flat
    )

def validate_efficiency():
    import gc
    from torch.nn import CrossEntropyLoss
    
    # Test configuration
    bsz, qlen, hd, vocab = 4, 4096, 4096, 128000
    X = torch.randn(bsz, qlen, hd, dtype=torch.bfloat16).cuda().requires_grad_(True)
    linear = nn.Linear(hd, vocab).cuda().bfloat16()
    labels = torch.randint(0, vocab, (bsz, qlen)).cuda()

    # Baseline measurement - FIXED DIMENSIONS
    torch.cuda.reset_peak_memory_stats()
    logits_normal = linear(X).float().view(-1, vocab)  # Reshape to [bsz*qlen, vocab]
    loss_normal = F.cross_entropy(logits_normal, labels.view(-1))  # Targets [bsz*qlen]
    loss_normal.backward()
    mem_normal = torch.cuda.max_memory_allocated()
    grad_normal = linear.weight.grad.clone()
    linear.zero_grad()
    
    # Our implementation - FIXED DIMENSIONS
    torch.cuda.reset_peak_memory_stats()
    loss_efficient = transformation_function(
        X.view(bsz, qlen, hd),  # Ensure original dimensions
        linear,
        labels.view(bsz, qlen)  # Ensure original dimensions
    )
    loss_efficient.backward()
    mem_efficient = torch.cuda.max_memory_allocated()
    grad_efficient = linear.weight.grad.clone()
    linear.zero_grad()

    # Criterion 1: VRAM reduction
    vram_reduction = (mem_normal - mem_efficient) / mem_normal
    print(f"VRAM reduction: {vram_reduction:.1%} (Target >50%)")
    
    # Criterion 2: No float32 upcast
    print(f"Using dtype: {loss_efficient.dtype} (Should be bfloat16)")
    
    # Criterion 3: CE Loss works
    loss_diff = torch.abs(loss_normal - loss_efficient).item()
    print(f"Loss difference: {loss_diff:.2e} (Should be <1e-3)")
    
    # Criterion 4: Other functions work
    try:
        # Test with MSE
        def mse_transform(X, linear, labels):
            return F.mse_loss(linear(X), torch.randn_like(linear(X)))
            
        loss_mse = MemoryEfficientLinear.apply(X, linear.weight, None, labels, mse_transform)
        loss_mse.backward()
        print("Other functions: ✓")
    except:
        print("Other functions: ✗")

    # Criterion 5: Dynamic chunk sizes
    try:
        for chunk_size in [4096, 8192, 16384]:
            MemoryEfficientLinear.apply(X, linear.weight, None, labels, chunk_size).backward()
        print("Dynamic chunks: ✓")
    except:
        print("Dynamic chunks: ✗")

    # Criterion 6: Gradient matching
    grad_error = torch.max(torch.abs(grad_normal - grad_efficient)).item()
    print(f"Gradient error: {grad_error:.2e} (Should be <1e-3)")

    # Cleanup
    del X, linear, labels
    gc.collect()
    torch.cuda.empty_cache()

# Run validation
validate_efficiency()


OutOfMemoryError: CUDA out of memory. Tried to allocate 7.81 GiB. GPU 0 has a total capacty of 44.42 GiB of which 1.78 GiB is free. Process 2982879 has 42.63 GiB memory in use. Of the allocated memory 37.40 GiB is allocated by PyTorch, and 4.72 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF